# Item B

### Carregamento dos dados

In [10]:
import torch
from torch.utils.data import DataLoader
from torchvision import transforms
from medmnist import BloodMNIST
import os

# Definir a transformação para converter as imagens em Tensores PyTorch [0.0, 1.0]
# Nota: O ToTensor() já divide automaticamente os pixels [0, 255] por 255.0
data_transform = transforms.Compose([
    transforms.ToTensor()
])

NUM_WORKERS = min(4, os.cpu_count())
print(f"NUM_WORKERS: {NUM_WORKERS}")

# Carregar os datasets oficiais do BloodMNIST (Treino, Validação e Teste)
# O parâmetro download=True baixa a base automaticamente caso não exista localmente
train_dataset = BloodMNIST(split='train', transform=data_transform, download=True)
val_dataset   = BloodMNIST(split='val', transform=data_transform, download=True)
test_dataset  = BloodMNIST(split='test', transform=data_transform, download=True)

# Criar os DataLoaders para gerenciar o mini-batching
BATCH_SIZE = 64

#train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
#val_loader   = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)


# gpu?
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"DEVICE: {DEVICE}")


train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True if DEVICE.type == 'cuda' else False,
    persistent_workers=True if NUM_WORKERS > 0 else False  # Mantém os processos vivos entre épocas
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True if DEVICE.type == 'cuda' else False,
    persistent_workers=True if NUM_WORKERS > 0 else False
)

NUM_WORKERS: 4
DEVICE: cuda


### Treino do Modelo

In [ ]:
import pandas as pd
import ml3.cnn_bc as cnn_bc


num_kernels_list=[8, 16, 32, 64, 128, 256]
kernel_sizes=[3, 5, 7]

results = cnn_bc.fit_cnn(
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    num_kernels_list=num_kernels_list,
    kernel_sizes=kernel_sizes,
    lr=1e-3,
    epochs=20,
    save_tag="_cnn_item_b"
)

# 3. Converter para DataFrame e visualizar estilo Pandas/Plotly
df_results = pd.DataFrame(results)
df_results.to_csv("item_b_cnn_validation_info.csv", index=False)



--- Testando CNN: num_kernels=8, kernel_size=3 ---

--- Testando CNN: num_kernels=8, kernel_size=5 ---

--- Testando CNN: num_kernels=8, kernel_size=7 ---

--- Testando CNN: num_kernels=16, kernel_size=3 ---

--- Testando CNN: num_kernels=16, kernel_size=5 ---

--- Testando CNN: num_kernels=16, kernel_size=7 ---

--- Testando CNN: num_kernels=32, kernel_size=3 ---

--- Testando CNN: num_kernels=32, kernel_size=5 ---

--- Testando CNN: num_kernels=32, kernel_size=7 ---

--- Testando CNN: num_kernels=64, kernel_size=3 ---

--- Testando CNN: num_kernels=64, kernel_size=5 ---

--- Testando CNN: num_kernels=64, kernel_size=7 ---

--- Testando CNN: num_kernels=128, kernel_size=3 ---

--- Testando CNN: num_kernels=128, kernel_size=5 ---

--- Testando CNN: num_kernels=128, kernel_size=7 ---

Melhor Configuração de CNN Encontrada: {'num_kernels': 128, 'kernel_size': 3}
Acurácia Balanceada na Validação: 0.9101


### Plotagem do Treino (Validação)

In [12]:
import pandas as pd

df_results = pd.read_csv("item_b_cnn_validation_info.csv")
df_sorted = df_results.sort_values(by="val_acc", ascending=False).reset_index(drop=True)

df_sorted = df_sorted.rename(columns={
    "num_kernels": "Kernels (K)",
    "kernel_size": "Tamanho (k)"
})

styled_df = df_sorted.style.background_gradient(
    cmap="Blues", subset=["val_acc"]
).format({
    "val_acc": "{:.4f}"
}).set_caption("<b>Progressão da Acurácia Balanceada na Validação da CNN</b>")

# Exibir a tabela no Notebook
styled_df

,Kernels (K),Tamanho (k),val_acc
0,128,3,0.9101
1,64,3,0.9088
2,128,5,0.9013
3,32,3,0.8960
4,32,5,0.8953
5,64,5,0.8924
6,16,3,0.8800
7,128,7,0.8733
8,16,5,0.8721
9,64,7,0.8684


In [13]:
import plotly.express as px

# Pivotear os dados para formato matricial (Linhas = K, Colunas = k)
pivot_df = df_results.pivot(index="num_kernels", columns="kernel_size", values="val_acc")

fig_heatmap = px.imshow(
    pivot_df,
    labels=dict(x="Tamanho do Kernel (k)", y="Quantidade de Kernels (K)", color="Acurácia Val"),
    x=[f"{k}x{k}" for k in pivot_df.columns],
    y=[str(K) for K in pivot_df.index],
    text_auto=".4f",
    color_continuous_scale="Blues",
    title="<b>Matriz de Acurácia de Validação da CNN (K x k)</b>"
)

fig_heatmap.update_layout(
    template="plotly_white",
    width=600,
    height=450
)

fig_heatmap.show()

### Teste